# Amazon Bedrock Agent Core Evaluations - Tutorial

This notebook provides a comprehensive guide to understanding and using **Amazon Bedrock Agent Core Evaluations** - the AWS managed service for evaluating AI agents.

## Table of Contents
1. [Introduction](#introduction)
2. [What is Agent Core Evaluations?](#what-is)
3. [Key Concepts](#key-concepts)
4. [Prerequisites & Setup](#setup)
5. [Architecture & How It Works](#architecture)
6. [Built-in Evaluators](#builtin-evaluators)
7. [Custom Evaluators](#custom-evaluators)
8. [On-Demand Evaluations](#ondemand)
9. [Online Evaluations](#online)
10. [Best Practices](#best-practices)

## Resources
- [Official Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html)
- [AgentCore Samples Repository](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/01-tutorials/07-AgentCore-evaluations)


## 1. Introduction <a id="introduction"></a>

**Amazon Bedrock Agent Core Evaluations** is an AWS managed service that automatically assesses the quality and performance of your AI agents using LLM-as-a-Judge techniques.

### What This Service Does

- **Automated Evaluation**: Uses powerful LLMs (like Claude) to evaluate agent interactions
- **Built-in Metrics**: Pre-configured evaluators for common quality dimensions
- **Custom Evaluators**: Create domain-specific evaluation criteria
- **Trace-Based**: Evaluates agents based on OpenTelemetry traces
- **Production Ready**: Supports both on-demand and online evaluation modes

### Why Use This Service?

- **Quality Assurance**: Ensure agents meet quality standards before deployment
- **Continuous Monitoring**: Track agent performance in production
- **Data-Driven Optimization**: Get quantitative metrics to guide improvements
- **Compliance**: Verify agents follow required behaviors and policies
- **Early Detection**: Catch performance regressions quickly

## 2. What is Agent Core Evaluations? <a id="what-is"></a>

Amazon Bedrock Agent Core Evaluations is part of the Bedrock Agent Core platform. It evaluates agents by analyzing their interaction traces.

### How It Works

```
┌─────────────────────┐
│   Your Agent        │
│ (Strands/LangGraph) │
└──────────┬──────────┘
           │
           │ Generates traces (OpenTelemetry)
           ▼
┌─────────────────────┐
│ AgentCore Runtime   │
│ + Observability     │
└──────────┬──────────┘
           │
           │ Stores traces
           ▼
┌─────────────────────┐
│   Trace Store       │
│ (span/trace/session)│
└──────────┬──────────┘
           │
           │ Evaluation Request
           ▼
┌─────────────────────┐      ┌──────────────────┐
│  AgentCore          │◄─────│  Evaluator LLM   │
│  Evaluations API    │      │  (Claude, etc)   │
└──────────┬──────────┘      └──────────────────┘
           │
           │ Scores & Reasoning
           ▼
┌─────────────────────┐
│  Evaluation Results │
│  (stored/exported)  │
└─────────────────────┘
```

### Key Components

1. **Agent**: Your Strands or LangGraph agent running on AgentCore Runtime
2. **Observability**: OpenTelemetry instrumentation capturing traces
3. **Evaluators**: Built-in or custom scoring logic
4. **Evaluation API**: AWS service for running evaluations via boto3
5. **Results**: Scores, reasoning, and metrics

### Evaluation Types

- **On-Demand**: Evaluate specific traces/sessions synchronously
- **Online**: Continuously evaluate production traffic automatically


## 3. Key Concepts <a id="key-concepts"></a>

### 3.1 Traces & Spans

Agent interactions are captured as **OpenTelemetry traces**:

- **Span**: Single operation (e.g., LLM call, tool invocation)
- **Trace**: Complete request/response cycle (multiple spans)
- **Session**: Multiple related traces over time

Evaluations analyze these traces to assess agent behavior.

### 3.2 Evaluators

An **evaluator** is scoring logic that judges agent performance:

**Built-in Evaluators** (Provided by AWS):
- ARN format: `arn:aws:bedrock-agentcore:::evaluator/Builtin.Helpfulness`
- Examples: Helpfulness, Relevance, Accuracy, Completeness
- Public and ready to use

**Custom Evaluators** (You create):
- ARN format: `arn:aws:bedrock-agentcore:region:account:evaluator/your-id`
- Domain-specific criteria
- Access controlled via IAM

### 3.3 Evaluation Configurations

A configuration defines:
- Which evaluator to use
- Target trace pattern (span/trace/session)
- Sampling rate (for online evaluations)
- Output destination (S3, CloudWatch)

### 3.4 Supported Agent Targets

- Amazon Bedrock Agent Core agents (Strands SDK)
- Amazon Bedrock Agent Core agents (LangGraph)
- Agents deployed on AgentCore Runtime
- Must have observability enabled (OpenTelemetry)

### 3.5 Service Quotas

| Resource | Limit |
|----------|-------|
| Evaluation configurations per region | 1,000 |
| Active online evaluations | 100 |
| Token throughput | 1M tokens/min |


## 4. Prerequisites & Setup <a id="setup"></a>

### 4.1 Prerequisites

Before using AgentCore Evaluations, you need:

1. **An Agent deployed on AgentCore Runtime**
   - Built with Strands SDK or LangGraph
   - Deployed to AgentCore Runtime

2. **Observability Enabled**
   - OpenTelemetry instrumentation configured
   - Traces being captured and stored

3. **At least one session with traces**
   - Agent has been invoked
   - Traces are available for evaluation

4. **AWS Credentials**
   - IAM permissions for bedrock-agentcore APIs
   - Access to evaluator ARNs

### 4.2 Install Required Libraries


In [ ]:
# Install AWS SDK and AgentCore evaluation toolkit
!pip install -q boto3 botocore bedrock-agentcore-starter-toolkit

# Install OpenTelemetry (if instrumenting agents locally)
!pip install -q opentelemetry-api opentelemetry-sdk aws-opentelemetry-distro

print("✅ Dependencies installed")

### 4.3 (Optional) Deploy a Simple Strands Agent on AgentCore Runtime

> **Skip this section** if you already have an agent deployed with observability enabled.

Below, we create a minimal Strands agent with two tools (`calculator` and `weather`) and deploy it to AgentCore Runtime. Observability (OpenTelemetry) is **automatically enabled** when running on AgentCore Runtime — no extra configuration needed.

In [ ]:
# Install Strands and AgentCore deployment dependencies
!pip install -q strands-agents strands-agents-tools bedrock-agentcore bedrock-agentcore-starter-toolkit

print("✅ Strands and AgentCore dependencies installed")

In [ ]:
%%writefile strands_eval_agent.py
from strands import Agent, tool
from strands_tools import calculator
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

@tool
def weather(city: str) -> str:
    """Get the current weather for a given city."""
    data = {
        "paris": "22°C, Sunny",
        "london": "15°C, Cloudy",
        "new york": "28°C, Clear",
        "tokyo": "18°C, Partly cloudy",
    }
    return data.get(city.lower(), f"No weather data for {city}")

model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You are a helpful assistant. You can do math and check the weather."
)

app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload):
    response = agent(payload.get("prompt"))
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

# Write requirements for the agent container
with open("agent_requirements.txt", "w") as f:
    f.write("strands-agents\nstrands-agents-tools\nboto3\nbedrock-agentcore\n")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_eval_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="agent_requirements.txt",
    region=region,
    agent_name="strands_eval_demo"
)

launch_result = agentcore_runtime.launch()

print(f"✅ Agent deployment initiated")
print(f"Agent ID: {launch_result.agent_id}")
print(f"Agent ARN: {launch_result.agent_arn}")

### 4.4 (Optional) Invoke the Agent to Generate a Trace

Once deployed, invoke the agent to generate at least one OpenTelemetry trace. The resulting **session ID** will be used for evaluations later.

In [ ]:
import time

# Wait for agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']

while status not in ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']:
    print(f"⏳ Status: {status} - waiting...")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

if status == 'READY':
    print("✅ Agent is ready!\n")

    # Invoke the agent — this generates an OpenTelemetry trace
    response = agentcore_runtime.invoke(
        {"prompt": "What is 15 * 7? Also, how is the weather in Paris?"}
    )
    print(f"Agent response:\n{response}")

    print(f"\n💡 Agent ID: {launch_result.agent_id}")
    print("💡 Find session IDs in CloudWatch > GenAI Observability > Bedrock Agentcore")
else:
    print(f"❌ Deployment failed: {status}")

### 4.5 Configure AWS SDK


In [ ]:
import boto3
import json
from datetime import datetime
from typing import Dict, List, Any
from bedrock_agentcore_starter_toolkit import Evaluation

# Configure AWS session
boto_session = boto3.Session(region_name='us-east-1')  # Change region as needed
region = boto_session.region_name

# Initialize evaluation client (from starter toolkit)
eval_client = Evaluation(region=region)

# Initialize boto3 clients (for runtime invocation and other operations)
agentcore_client = boto_session.client('bedrock-agentcore')

print(f"✅ AWS Session configured for region: {region}")
print(f"✅ Evaluation client initialized")

### 4.6 Verify Prerequisites


In [ ]:
# Example: Check if you have access to evaluators
# Replace with actual API calls based on AWS documentation

def check_evaluator_access():
    """
    Verify access to built-in evaluators
    """
    try:
        # Example built-in evaluator ARN
        evaluator_arn = "arn:aws:bedrock-agentcore:::evaluator/Builtin.Helpfulness"
        
        # In practice, you'd call an API to verify access
        # response = agentcore_client.describe_evaluator(evaluatorArn=evaluator_arn)
        
        print(f"✅ Access verified for evaluator: {evaluator_arn}")
        return True
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return False

check_evaluator_access()

print("💡 Make sure you've:")
print("  1. Deployed an agent on AgentCore Runtime")
print("  2. Enabled observability")
print("  3. Generated at least one session")


## 5. Architecture & How It Works <a id="architecture"></a>

### 5.1 The Evaluation Flow

1. **Agent Execution**
   - Your agent processes user requests
   - OpenTelemetry captures spans (LLM calls, tool usage, reasoning)
   - Traces are sent to AgentCore Runtime observability

2. **Trace Storage**
   - Traces are stored with unique IDs (span ID, trace ID, session ID)
   - Metadata includes timestamps, latencies, inputs/outputs

3. **Evaluation Request**
   - You submit an evaluation request via boto3
   - Specify: evaluator ARN, trace/session ID, configuration

4. **LLM-as-Judge**
   - AWS invokes an LLM (like Claude) as the evaluator
   - LLM reviews the trace data against criteria
   - Generates scores and reasoning

5. **Results**
   - Scores (typically 1-10 or 0-1 scale)
   - Reasoning explaining the score
   - Stored in specified location (S3, DynamoDB)

### 5.2 LLM-as-a-Judge Pattern

The service uses a powerful LLM to evaluate your agent:

```python
# Conceptual flow (not actual API)
evaluation_prompt = f"""
You are an expert evaluator. Review this agent interaction:

User Query: {user_input}
Agent Response: {agent_response}
Tool Calls: {tools_used}
Reasoning: {agent_reasoning}

Evaluate on: {evaluation_criteria}

Provide:
1. Score (1-10)
2. Reasoning for the score
3. Specific strengths and weaknesses
"""

# LLM processes the prompt and returns structured evaluation
```

The evaluator LLM acts as an objective judge of your agent's performance.


## 6. Built-in Evaluators <a id="builtin-evaluators"></a>

AWS provides pre-built evaluators for common quality dimensions.

### 6.1 Available Built-in Evaluators

| Evaluator | Name | Description |
|-----------|------|-------------|
| **Goal Success Rate** | `Builtin.GoalSuccessRate` | Measures whether the agent achieved the user's goal |
| **Correctness** | `Builtin.Correctness` | Verifies factual correctness of the response |
| **Tool Selection Accuracy** | `Builtin.ToolSelectionAccuracy` | Checks if the agent selected the right tools |
| **Tool Parameter Accuracy** | `Builtin.ToolParameterAccuracy` | Checks if tool parameters were set correctly |

> **Note**: Check the [official documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations-builtin.html) for the complete and up-to-date list.

### Retrieving built-in evaluators
Let's now retrieve the available built-in evaluators to understand where they can be used. The list_evaluators() function can help with it.

In [ ]:
available_evaluators = eval_client.list_evaluators()
available_evaluators

### 6.2 Using Built-in Evaluators

Built-in evaluators are:
- **Public**: No IAM permissions needed beyond basic API access
- **Pre-configured**: Ready to use with standard criteria
- **Maintained by AWS**: Updated and improved over time

In [ ]:
# Using a built-in evaluator

# Replace with your agent and session IDs
agent_id = "your-agent-id"                              # from launch_result.agent_id
session_id = "your-session-id"     # from CloudWatch or invocation

# Run evaluation with a built-in evaluator
results = eval_client.run(
    agent_id=agent_id,
    session_id=session_id,
    evaluators=["Builtin.GoalSuccessRate"]
)

# Display results
for result in results.results:
    print(f"Evaluator: {result.evaluator_name}")
    print(f"Result: {result.label} ({result.value})")
    print(f"Explanation:\n{result.explanation}")
    print(f"Token Usage: {result.token_usage}")
    print("=" * 60)

## 7. Custom Evaluators <a id="custom-evaluators"></a>

Create custom evaluators for domain-specific requirements.

### 7.1 When to Create Custom Evaluators

- **Domain-Specific Criteria**: Evaluate industry-specific behaviors
- **Compliance Requirements**: Check regulatory adherence
- **Custom Quality Standards**: Enforce your organization's standards
- **Specialized Metrics**: Measure unique performance aspects

### 7.2 Custom Evaluator Structure

A custom evaluator uses the **LLM-as-a-Judge** pattern and defines:
1. **Model**: Which LLM to use as the judge (e.g., Claude Sonnet)
2. **Instructions**: What to evaluate and how
3. **Ratings**: Scoring scale with named tiers and definitions
4. **Level**: Granularity — `TRACE` (single request) or `SESSION` (full conversation)

### 7.3 Example: E-Commerce Compliance Evaluator

In [ ]:
# Custom evaluator configuration using the LLM-as-a-Judge format

eval_config = {
    "llmAsAJudge": {
        "modelConfig": {
            "bedrockEvaluatorModelConfig": {
                "modelId": "global.anthropic.claude-sonnet-4-5-20250929-v1:0",
                "inferenceConfig": {
                    "maxTokens": 500,
                    "temperature": 1.0
                }
            }
        },
        "instructions": (
            "You are evaluating an e-commerce agent's interaction for policy compliance. "
            "Assess whether the agent clearly communicates pricing, mentions return policies "
            "when relevant, provides accurate inventory information, and handles personal "
            "data appropriately. Focus on the substance of the response, not style.\n\n"
            "Context: {context}\n"
            "Candidate Response: {assistant_turn}"
        ),
        "ratingScale": {
            "numerical": [
                {
                    "value": 1,
                    "label": "Fully Compliant",
                    "definition": "Agent addresses all compliance criteria: transparent pricing, return policy mentioned, accurate inventory, proper data handling."
                },
                {
                    "value": 0.75,
                    "label": "Mostly Compliant",
                    "definition": "Agent meets most compliance criteria with minor omissions that don't significantly impact the customer experience."
                },
                {
                    "value": 0.50,
                    "label": "Partially Compliant",
                    "definition": "Agent misses notable compliance areas, such as omitting shipping costs or not mentioning return policy when relevant."
                },
                {
                    "value": 0.25,
                    "label": "Non-Compliant",
                    "definition": "Agent has significant compliance gaps, such as misleading pricing or incorrect inventory information."
                },
                {
                    "value": 0,
                    "label": "Critical Violation",
                    "definition": "Agent provides completely incorrect information, mishandles personal data, or actively misleads the customer."
                }
            ]
        }
    }
}

print("Custom Evaluator Configuration:")
print("=" * 60)
print(f"Model: {eval_config['llmAsAJudge']['modelConfig']['bedrockEvaluatorModelConfig']['modelId']}")
print(f"Instructions: {eval_config['llmAsAJudge']['instructions'][:80]}...")
print(f"\nRating Scale ({len(eval_config['llmAsAJudge']['ratingScale']['numerical'])} tiers):")
for rating in eval_config['llmAsAJudge']['ratingScale']['numerical']:
    print(f"  {rating['label']} ({rating['value']}): {rating['definition'][:60]}...")

### 7.4 Creating the Custom Evaluator

In [ ]:
# Create the custom evaluator
custom_evaluator = eval_client.create_evaluator(
    name="ecommerce_compliance",
    level="TRACE",
    description="Evaluates e-commerce agent compliance with policies",
    config=eval_config
)

evaluator_id = custom_evaluator['evaluatorId']
print(f"✅ Custom evaluator created")
print(f"Evaluator ID: {evaluator_id}")

# You can now use this evaluator_id in eval_client.run()
# alongside built-in evaluators

## 8. On-Demand Evaluations <a id="ondemand"></a>

**On-demand evaluations** let you evaluate specific agent interactions synchronously.

### 8.1 When to Use On-Demand

- **Troubleshooting**: Investigate specific customer issues
- **Validation**: Test fixes or improvements
- **Historical Analysis**: Evaluate past interactions
- **Development**: Test agent behavior during development
- **Spot Checking**: Sample production interactions

### 8.2 How On-Demand Works

1. Identify the trace/session you want to evaluate
2. Choose an evaluator (built-in or custom)
3. Submit synchronous evaluation request
4. Receive results immediately

### 8.3 Evaluation Targets

You can evaluate at different granularities:
- **Span ID**: Single operation (one LLM call, one tool use)
- **Trace ID**: Complete request/response cycle
- **Session ID**: Multiple related interactions

### 8.4 Example: On-Demand Evaluation


In [ ]:
# On-demand evaluation with multiple evaluators

# Replace with your IDs
agent_id = "your-agent-id"
session_id = "your-session-id"

# Run multiple evaluators at once
results = eval_client.run(
    agent_id=agent_id,
    session_id=session_id,
    evaluators=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolSelectionAccuracy",
        "Builtin.ToolParameterAccuracy"
    ]
)

# Display all results
for result in results.results:
    print(f"Evaluator: {result.evaluator_name}")
    print(f"Result: {result.label} ({result.value})")
    print(f"Explanation:\n{result.explanation}\n")
    print(f"Token Usage: {result.token_usage}")
    print(f"Context: {result.context}")
    print("=" * 60)

### 8.5 Batch On-Demand Evaluations

Evaluate multiple sessions programmatically:


In [ ]:
# Batch evaluation across multiple sessions

def batch_evaluate_sessions(
    agent_id: str,
    session_ids: List[str],
    evaluators: List[str]
):
    """Evaluate multiple sessions in batch."""
    all_results = []

    print(f"Evaluating {len(session_ids)} sessions with {len(evaluators)} evaluators...")
    print("=" * 60)

    for i, sid in enumerate(session_ids, 1):
        print(f"\n[{i}/{len(session_ids)}] Session: {sid}")
        try:
            results = eval_client.run(
                agent_id=agent_id,
                session_id=sid,
                evaluators=evaluators
            )
            for result in results.results:
                print(f"  {result.evaluator_name}: {result.label} ({result.value})")
                all_results.append({
                    'session_id': sid,
                    'evaluator': result.evaluator_name,
                    'label': result.label,
                    'value': result.value,
                    'explanation': result.explanation
                })
        except Exception as e:
            print(f"  ❌ Error: {str(e)}")
            all_results.append({'session_id': sid, 'error': str(e)})

    return all_results

# Example usage (uncomment with real IDs)
# results = batch_evaluate_sessions(
#     agent_id="your-agent-id",
#     session_ids=["session-1", "session-2", "session-3"],
#     evaluators=["Builtin.GoalSuccessRate", "Builtin.Correctness"]
# )

print("💡 Batch evaluations help analyze multiple interactions efficiently")

## 9. Online Evaluations <a id="online"></a>

**Online evaluations** automatically evaluate agent interactions in production as they occur.

### 9.1 When to Use Online

- **Production Monitoring**: Continuously track agent quality
- **Regression Detection**: Catch performance drops quickly
- **Quality Metrics**: Generate ongoing quality dashboards
- **Alerting**: Trigger alerts when scores drop
- **A/B Testing**: Compare different agent versions

### 9.2 How Online Works

1. Create an online evaluation configuration
2. Specify sampling rate (e.g., evaluate 100% of traffic)
3. Choose evaluator(s) — built-in and/or custom
4. AWS automatically evaluates sampled interactions
5. Results appear in CloudWatch GenAI Observability

### 9.3 Create Online Evaluation Configuration

In [ ]:
# Create an online evaluation configuration

agent_id = "your-agent-id"  # Replace with your agent ID

response = eval_client.create_online_config(
    agent_id=agent_id,
    config_name="eval_demo_online",
    sampling_rate=100,  # Percentage of traces to evaluate (1-100)
    evaluator_list=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolParameterAccuracy",
        "Builtin.ToolSelectionAccuracy",
        # evaluator_id  # Uncomment to include your custom evaluator
    ],
    config_description="Online evaluation for demo agent",
    auto_create_execution_role=True
)

online_config_id = response['onlineEvaluationConfigId']
print(f"✅ Online evaluation configured")
print(f"Configuration ID: {online_config_id}")

### 9.4 Viewing Online Evaluation Configuration

In [ ]:
# Retrieve online evaluation configuration details

config_details = eval_client.get_online_config(config_id=online_config_id)
config_details

### 9.5 Generate Traffic for Online Evaluation

With online evaluation active, every agent invocation (based on sampling rate) is automatically evaluated. Let's invoke the agent a few times to generate evaluation data.

In [ ]:
import boto3

agentcore_invoke_client = boto3.client('bedrock-agentcore', region_name=region)
# Dynamically retrieve account ID to avoid hardcoding
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']
agent_arn = f"arn:aws:bedrock-agentcore:{region}:{account_id}:runtime/{agent_id}"

def invoke_agent_runtime(agent_arn, prompt):
    """Invoke the agent and display the response."""
    response = agentcore_invoke_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        qualifier="DEFAULT",
        payload=json.dumps({"prompt": prompt})
    )
    # Parse response
    events = []
    for event in response.get("response", []):
        events.append(event)
    if events:
        result = json.loads(events[0].decode("utf-8"))
        print(f"Prompt: {prompt}")
        print(f"Response: {result}\n")
    return response

# Send a few queries to generate evaluation data
prompts = [
    "How much is 7+9+10*2?",
    "How is the weather in London?",
    "How much is 20% of 300?",
    "What is the weather in Tokyo?",
]

for prompt in prompts:
    invoke_agent_runtime(agent_arn, prompt)

print("💡 Online evaluations will automatically process these traces")

### 9.6 Viewing Online Evaluation Results


In [ ]:
# Example: Retrieve and analyze online evaluation results

def get_evaluation_metrics(
    config_id: str,
    start_time: datetime = None,
    end_time: datetime = None
):
    """
    Retrieve online evaluation metrics for analysis
    
    Args:
        config_id: Configuration ID
        start_time: Start of time range
        end_time: End of time range
    
    Returns:
        Metrics dictionary
    """
    try:
        # Query CloudWatch or S3 for results
        # This is conceptual - actual implementation depends on output format
        
        # Example: Query from CloudWatch
        cloudwatch = session.client('cloudwatch')
        
        response = cloudwatch.get_metric_statistics(
            Namespace='AWS/BedrockAgentCore/Evaluations',
            MetricName='EvaluationScore',
            Dimensions=[
                {'Name': 'ConfigurationId', 'Value': config_id}
            ],
            StartTime=start_time or datetime.now() - timedelta(days=7),
            EndTime=end_time or datetime.now(),
            Period=3600,  # 1 hour
            Statistics=['Average', 'Minimum', 'Maximum', 'SampleCount']
        )
        
        datapoints = response['Datapoints']
        
        if datapoints:
            avg_score = sum(dp['Average'] for dp in datapoints) / len(datapoints)
            min_score = min(dp['Minimum'] for dp in datapoints)
            max_score = max(dp['Maximum'] for dp in datapoints)
            total_evals = sum(dp['SampleCount'] for dp in datapoints)
            
            print(f"Online Evaluation Metrics")
            print("=" * 60)
            print(f"Configuration: {config_id}")
            print(f"Period: {start_time} to {end_time}")
            print(f"\nResults:")
            print(f"  Total Evaluations: {total_evals}")
            print(f"  Average Score: {avg_score:.2f}/10")
            print(f"  Min Score: {min_score:.2f}/10")
            print(f"  Max Score: {max_score:.2f}/10")
            
            return {
                'average': avg_score,
                'min': min_score,
                'max': max_score,
                'count': total_evals
            }
        else:
            print("No data available for specified period")
            return None
            
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return None

# Example usage
from datetime import timedelta
metrics = get_evaluation_metrics(
     config_id="eval_demo_online-exaaBX6WFH",
     start_time=datetime.now() - timedelta(days=1)
 )

print("💡 Track evaluation metrics over time to identify trends")


## 10. Best Practices <a id="best-practices"></a>

### 10.1 Choosing Evaluation Strategy

**Use On-Demand when:**
- Developing and testing agents
- Investigating specific issues
- Validating fixes
- Analyzing historical data
- Cost-sensitive scenarios

**Use Online when:**
- Monitoring production agents
- Detecting regressions
- Tracking quality trends
- A/B testing agent versions
- Need continuous metrics

### 10.2 Sampling Strategy

For online evaluations, choose sampling rates wisely:

| Traffic Volume | Recommended Sampling | Rationale |
|----------------|---------------------|-----------|
| < 1,000/day | 100% | Low volume, evaluate everything |
| 1,000-10,000/day | 10-20% | Balance cost and coverage |
| 10,000-100,000/day | 1-5% | Statistically significant sample |
| > 100,000/day | 0.1-1% | Representative sampling |

**Factors to consider:**
- Cost (LLM evaluator calls)
- Statistical significance needed
- Criticality of the application
- Service quota limits


In [ ]:
# Example: Calculate appropriate sampling rate

def calculate_sampling_rate(daily_interactions: int, target_evaluations: int = 100):
    """
    Calculate sampling rate to achieve target number of evaluations
    
    Args:
        daily_interactions: Expected daily interaction volume
        target_evaluations: Target number of evaluations per day
    
    Returns:
        Recommended sampling rate
    """
    if daily_interactions <= target_evaluations:
        sampling_rate = 1.0  # Evaluate everything
    else:
        sampling_rate = target_evaluations / daily_interactions
    
    # Ensure within bounds
    sampling_rate = max(0.001, min(1.0, sampling_rate))
    
    estimated_daily = daily_interactions * sampling_rate
    estimated_monthly_cost = estimated_daily * 30 * 0.01  # Rough cost estimate
    
    print(f"Sampling Rate Recommendation")
    print("=" * 60)
    print(f"Daily Interactions: {daily_interactions:,}")
    print(f"Target Evaluations/Day: {target_evaluations:,}")
    print(f"\nRecommended Sampling Rate: {sampling_rate*100:.2f}%")
    print(f"\nEstimated Results:")
    print(f"  Evaluations/Day: ~{estimated_daily:,.0f}")
    print(f"  Evaluations/Month: ~{estimated_daily*30:,.0f}")
    print(f"  Estimated Monthly Cost: ~${estimated_monthly_cost:.2f}")
    print(f"\n💡 Adjust based on your quality requirements and budget")
    
    return sampling_rate

# Examples
calculate_sampling_rate(daily_interactions=5000, target_evaluations=500)
calculate_sampling_rate(daily_interactions=100000, target_evaluations=1000)


### 10.3 Evaluation Design

**1. Choose Appropriate Evaluators**
- Use built-in evaluators for general quality
- Create custom evaluators for domain-specific needs
- Combine multiple evaluators for comprehensive assessment

**2. Set Clear Thresholds**
```python
thresholds = {
    'excellent': 9.0,    # Agent exceeds expectations
    'good': 7.0,         # Agent meets requirements
    'acceptable': 5.0,   # Agent needs improvement
    'critical': 3.0      # Agent requires immediate attention
}
```

**3. Test Evaluators First**
- Run on-demand on sample sessions
- Verify evaluator behavior matches expectations
- Iterate on custom evaluator prompts
- Compare with human judgments

**4. Monitor Trends, Not Just Scores**
- Track score changes over time
- Look for patterns (time of day, user types)
- Correlate with other metrics (latency, cost)

### 10.4 Cost Optimization


In [ ]:
# Cost optimization strategies

cost_optimization = {
    "sampling": {
        "strategy": "Sample intelligently",
        "details": [
            "Start with 1-5% for high-volume applications",
            "Increase sampling for critical interactions",
            "Use stratified sampling (test all error cases, sample success cases)"
        ],
        "savings": "80-95%"
    },
    "caching": {
        "strategy": "Cache similar evaluations",
        "details": [
            "Don't re-evaluate identical queries",
            "Cache evaluations for common patterns",
            "Use deterministic session IDs for testing"
        ],
        "savings": "20-40%"
    },
    "targeting": {
        "strategy": "Evaluate strategically",
        "details": [
            "Focus on high-value interactions",
            "Evaluate new features more heavily",
            "Reduce evaluation of stable components"
        ],
        "savings": "30-50%"
    },
    "batching": {
        "strategy": "Batch on-demand evaluations",
        "details": [
            "Evaluate multiple sessions together",
            "Schedule during off-peak hours",
            "Use asynchronous evaluation when possible"
        ],
        "savings": "10-20%"
    }
}

print("Cost Optimization Strategies")
print("=" * 60)
for strategy_name, strategy_info in cost_optimization.items():
    print(f"\n{strategy_name.upper()}: {strategy_info['strategy']}")
    print(f"Potential Savings: {strategy_info['savings']}")
    print("Details:")
    for detail in strategy_info['details']:
        print(f"  • {detail}")

print("\n💡 Combine strategies for maximum cost efficiency")


### 10.5 Integration with Development Workflow

**Development Phase:**
1. Use on-demand evaluations extensively
2. Test with diverse scenarios
3. Iterate based on evaluation feedback
4. Establish baseline scores

**Staging Phase:**
1. Enable online evaluation (high sampling)
2. Run regression tests before deployment
3. Compare scores to baseline
4. Set up alerting

**Production Phase:**
1. Continue online evaluation (lower sampling)
2. Monitor dashboards and alerts
3. Investigate score drops immediately
4. Use on-demand for troubleshooting

### 10.6 Security & Compliance

**Data Privacy:**
- Evaluations may log user inputs and agent responses
- Configure data retention policies
- Use encryption for sensitive data
- Review IAM policies regularly

**IAM Best Practices:**
```python
# Example: Least-privilege IAM policy
# NOTE: Scope resources to specific regions and agent IDs.
# The example below uses wildcards for illustration — in production,
# replace with your actual region, account ID, and agent ID.
iam_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateEvaluation",
                "bedrock-agentcore:GetEvaluation",
                "bedrock-agentcore:ListEvaluations"
            ],
            "Resource": [
                "arn:aws:bedrock-agentcore:<region>:<account-id>:agent/<agent-id>",
                "arn:aws:bedrock-agentcore:::evaluator/Builtin.*"
            ]
        }
    ]
}
```

**Compliance Considerations:**
- Document evaluation criteria
- Maintain audit trails
- Regular review of custom evaluators
- Compliance-specific evaluators for regulated industries

> **Note on IAM roles:** This notebook uses `auto_create_execution_role=True` for convenience. In production, create and manage IAM roles explicitly with least-privilege permissions scoped to your specific agents and resources. The auto-created roles are intended for sample/demo purposes only.


## 11. Summary

### What You Learned

In this tutorial, you learned about **Amazon Bedrock Agent Core Evaluations**:

✅ **What it is**: AWS managed service for evaluating agents using LLM-as-a-Judge

✅ **Key Concepts**: 
   - Trace-based evaluation (spans, traces, sessions)
   - Built-in and custom evaluators
   - On-demand vs online evaluation modes

✅ **Implementation**:
   - Using boto3 SDK to access the service
   - Creating and using evaluators
   - Running on-demand evaluations
   - Setting up online monitoring

✅ **Best Practices**:
   - Choosing the right evaluation strategy
   - Optimizing costs through sampling
   - Integrating into development workflows
   - Security and compliance considerations

### Next Steps

1. **Get Started**:
   - Deploy an agent on AgentCore Runtime
   - Enable observability (OpenTelemetry)
   - Run your first on-demand evaluation

2. **Experiment**:
   - Try different built-in evaluators
   - Create a custom evaluator for your domain
   - Compare evaluation results

3. **Production**:
   - Set up online evaluation monitoring
   - Configure alerts and dashboards
   - Establish quality baselines
   - Integrate into CI/CD

4. **Optimize**:
   - Refine sampling strategies
   - Improve custom evaluators
   - Monitor cost and adjust
   - Track trends over time

### Resources

- **Documentation**: [AgentCore Evaluations Guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html)
- **Code Examples**: [AgentCore Samples Repository](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/01-tutorials/07-AgentCore-evaluations)
- **API Reference**: [Bedrock AgentCore API](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/)
- **AWS Support**: [re:Post Community](https://repost.aws/)

### Important Notes

⚠️ **This tutorial focuses on the AWS managed service** - Amazon Bedrock Agent Core Evaluations accessed via boto3.

⚠️ **API examples are conceptual** - The actual API methods and parameters should be verified against the official AWS documentation as the service may have different naming conventions.

⚠️ **Prerequisites are required** - You must have an agent deployed on AgentCore Runtime with observability enabled before you can run evaluations.

---

**Happy Evaluating!** 🚀

Building reliable AI agents requires systematic evaluation. Use this service to ensure your agents meet quality standards and perform well in production.
